# current Skill 지표만들기 (PCA결합 - 현재 폼 점수)

### 데이터 정제
* 미드랑 정글 둘 다 있으니 미드만 추출함 

In [3]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# =========================
# LOAD CSV
# =========================

scores = pd.read_csv(
    "./오라클데이터/full_performance_pca_scores.csv"
)

# MID만 사용
scores = scores[
    scores["position"] == "mid"
].copy()

variance = pd.read_csv(
    "./오라클데이터/full_performance_pca_explained_variance.csv"
)

# =========================
# 사용할 PC 개수
# =========================

TOP_PCS = 4

# =========================
# variance 기반 weight
# =========================

weights = (
    variance
    .head(TOP_PCS)
    ["explained_variance_ratio"]
)

weights = weights / weights.sum()

# 확인
print(weights)

# =========================
# 사용할 PC 컬럼
# =========================

pc_columns = [
    f"PC{i}"
    for i in range(1, TOP_PCS + 1)
]

# =========================
# CurrentSkill 계산
# =========================

scores["CurrentSkill_raw"] = 0

for pc, weight in zip(
    pc_columns,
    weights
):
    scores["CurrentSkill_raw"] += (
        scores[pc] * weight
    )

# =========================
# 0~100 정규화
# =========================

scaler = MinMaxScaler(
    feature_range=(0, 100)
)

scores["CurrentSkill"] = scaler.fit_transform(
    scores[["CurrentSkill_raw"]]
)

# =========================
# 결과 확인
# =========================

result = (
    scores[
        [
            "playername",
            "patch",
            "CurrentSkill",
            "PC1",
            "PC2",
            "PC3",
            "PC4",
        ]
    ]
    .sort_values(
        "CurrentSkill",
        ascending=False
    )
)

display(result.head(30))

0    0.472525
1    0.211860
2    0.173060
3    0.142554
Name: explained_variance_ratio, dtype: float64


,playername,patch,CurrentSkill,PC1,PC2,PC3,PC4
304,Chovy,14.03,100.000000,7.321784,-1.755341,1.312899,0.003686
351,Chovy,14.12,97.293544,7.263833,-0.765165,-0.148594,-0.740731
335,Chovy,14.06,96.848922,6.422779,-1.320753,2.329508,-0.339163
512,Chovy,16.01,96.744716,4.754193,4.679966,0.561814,-1.628147
315,Chovy,14.04,94.310987,6.579345,-2.485746,3.393110,-1.579976
374,Chovy,14.14,91.708325,6.653766,0.994421,-1.353566,-2.428013
449,Chovy,15.08,90.613980,5.672537,0.556807,-1.734088,1.435728
361,Chovy,14.13,89.546712,6.608390,0.487879,-0.564673,-3.472221
496,Chovy,15.16,88.923394,4.473408,1.168305,0.459334,1.064840
326,Chovy,14.05,88.075255,4.878836,1.220190,0.957725,-1.349513


In [4]:
# result값도 csv로 저장
result.to_csv(
    "./오라클데이터/current_skill_scores.csv",
    index=False
)